<a href="https://colab.research.google.com/github/INDHUJA007-HUB/indhuja-day15-workshop/blob/main/Day9/MiniProject_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
df=pd.read_csv("/content/drive/MyDrive/DAY 1/student_performance.csv")

In [5]:
## Install the dependices

!pip install groq -q  ## uses to call the LLM

print("Imported groq")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00
Imported groq


In [6]:
import sqlite3
import os  # create an environment (communicate with os )
from groq import Groq
import re   # to get regular format
print("All libraries imported successfully")

All libraries imported successfully


In [7]:
os.environ["GROQ_API_KEY"] = "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"
# we store the groq_api in safe environment is environ

In [8]:
## create an client for groq API key to for communication

client=Groq(api_key=os.environ["GROQ_API_KEY"])

### insert the model

MODEL="llama-3.1-8b-instant"

print("groq client initalized")
print(f'Using model:{MODEL}')

groq client initalized
Using model:llama-3.1-8b-instant


In [9]:
print(df.columns)

Index(['student_id', 'name', 'age', 'gender', 'department', 'semester',
       'math_score', 'science_score', 'english_score', 'programming_score',
       'attendance_percentage', 'city', 'admission_year'],
      dtype='object')


In [10]:

import io ## input/output operations

csv_data=pd.read_csv("/content/drive/MyDrive/DAY 1/student_performance.csv")
print(csv_data.head(5))

   student_id          name  age  gender        department  semester  \
0        1001  Aarav Sharma   19    Male  Computer Science         2   
1        1002   Priya Patel   20  Female  Computer Science         2   
2        1003   Rohit Verma   19    Male       Electronics         2   
3        1004   Sneha Reddy   20  Female        Mechanical         2   
4        1005    Arjun Nair   19    Male  Computer Science         2   

   math_score  science_score  english_score  programming_score  \
0          85             78             72                 91   
1          76             82             88                 79   
2          65             74             61                 55   
3          70             80             75                 48   
4          92             88             81                 95   

   attendance_percentage       city  admission_year  
0                     92     Mumbai            2023  
1                     87  Ahmedabad            2023  
2       

In [11]:
##create SQLITE DB

conn=sqlite3.connect("college.db")
#create an connection to the college db with sqlite3

csv_data.to_sql("students",conn,if_exists="replace",index=False)
# df.to_sql() converts a pandas df into sql table

print("db created")
test_df = pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)
print(f'Verification {test_df['total_rows'][0]} rows in database')

db created
Verification 30 rows in database


In [12]:
## function to create schema get

def get_schema(conn,table_name="students"):

  """ This is your description for you"""
  cursor = conn.cursor() #cursor is an connection for SQLITE used to execute SQL commands
  cursor.execute(f'PRAGMA table_info ({table_name})') # the special python function PRAGMA table_info is used to find the column name
  columns=cursor.fetchall() # fetches all the result of columns from prev ones
  schema_lines = [f'Table:{table_name}']
  schema_lines.append("Columns:") ## it stors the tablename in the above format

  for col in columns:
    schema_lines.append(f"-{col[1]}({col[2]})")

  cursor.execute(f'SELECT * FROM students LIMIT 3')

  sample_rows=cursor.fetchall();
  schema_lines.append("\n Sample rows{first 3}")
  for row in sample_rows:
    schema_lines.append(f"{row}")
  return "\n".join(schema_lines)

#.join():join all lines with newline character
schema=get_schema(conn)
print(schema)


  # the o/p will give the structure of DB

Table:students
Columns:
-student_id(INTEGER)
-name(TEXT)
-age(INTEGER)
-gender(TEXT)
-department(TEXT)
-semester(INTEGER)
-math_score(INTEGER)
-science_score(INTEGER)
-english_score(INTEGER)
-programming_score(INTEGER)
-attendance_percentage(INTEGER)
-city(TEXT)
-admission_year(INTEGER)

 Sample rows{first 3}
(1001, 'Aarav Sharma', 19, 'Male', 'Computer Science', 2, 85, 78, 72, 91, 92, 'Mumbai', 2023)
(1002, 'Priya Patel', 20, 'Female', 'Computer Science', 2, 76, 82, 88, 79, 87, 'Ahmedabad', 2023)
(1003, 'Rohit Verma', 19, 'Male', 'Electronics', 2, 65, 74, 61, 55, 78, 'Delhi', 2023)


In [13]:
### function to generate SQL

def generate_sql(user_question,schema_text,client,model):
  """ this is an generate_sql function
  user_question:typed by the user
  schema_text: it is the response from the llm """
  #define system prompt
  system_prompt=f"""You are an expert in natural language processing and SQL, with 10 years of experience.
                  Your task is to accurately translate natural language questions into valid SQLite SQL queries
                  based on the provided database schema.
  {schema_text}

Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or text — only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is: students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.

"""

  ## call the groq API
  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          # role: "system" means this is the instructions/context message
          #context: the actual instruction text
          {"role": "user", "content": user_question},
      ],

    temperature = 0.0  # for accurate value
  )

  sql_query=response.choices[0].message.content.strip()

  ## response.choices it says which choices needs to print because it has many choices
  # strip() = it uses to removes spaces
  return sql_query

# test: generate SQL for simple question
question="give the count of female and males students"
print(f"Question:{question}")
print("\nGenerating SQL....")

sql=generate_sql(question,schema,client,MODEL)
print(f'SQL:{sql}')

Question:give the count of female and males students

Generating SQL....
SQL:SELECT COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count, 
       COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count 
FROM students


In [14]:
def get_response(user_question):
  print(f"User Question: {user_question}")
  # Generate SQL query from the user's question
  sql_query = generate_sql(user_question, schema, client, MODEL)
  print(f"Generated SQL: {sql_query}")

  # Execute the SQL query and fetch results
  try:
    result_df = pd.read_sql_query(sql_query, conn)
    print("Query Result:")
    display(result_df)
  except Exception as e:
    print(f"Error executing SQL query: {e}")

# Example usage:
get_response("What are the names of all students?")

User Question: What are the names of all students?
Generated SQL: SELECT name FROM students
Query Result:


,name
0,Aarav Sharma
1,Priya Patel
2,Rohit Verma
3,Sneha Reddy
4,Arjun Nair
5,Meera Joshi
6,Kiran Kumar
7,Divya Singh
8,Rahul Mishra
9,Ananya Das
